In [1]:
try:
    spark.stop()
except:
    pass

from pyspark.sql import SparkSession

from pyspark.sql.functions import (
    sum as F_sum,
    avg as F_avg,
    min as F_min,
    max as F_max,
    countDistinct,
    count as F_count
)

spark = (
    SparkSession.builder
    .appName("lab2_etl_470177")
    .master("local[*]")
    .config("spark.jars", "/home/470177/postgresql-jdbc.jar")
    .config("spark.driver.extraClassPath", "/home/470177/postgresql-jdbc.jar")
    .config("spark.executor.extraClassPath", "/home/470177/postgresql-jdbc.jar")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")
spark

In [2]:
from pyspark.sql.functions import (
    col, to_date, sum as F_sum, avg as F_avg,
    countDistinct, min as F_min, max as F_max
)

from pymongo import MongoClient
import clickhouse_connect

POSTGRES_URL = "jdbc:postgresql://postgres:5432/bigdata_lab"
POSTGRES_PROPS = {"user": "postgres", "password": "postgres", "driver": "org.postgresql.Driver"}

MONGO_URI = "mongodb://mongo:27017"

CLICKHOUSE_HOST = "clickhouse"
CLICKHOUSE_PORT = 8123
CLICKHOUSE_USER = "labuser"
CLICKHOUSE_PASS = "labpass"

In [3]:
csv_path = "/data/all_mock_data.csv"

df_all = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .option ("multiline", "true")
    .option("quote", "\"")
    .option("escape", "\"")
    .csv(csv_path)
)

print(df_all.count(), len(df_all.columns))
df_all.show(5, truncate=False)

df_all.write \
    .mode("overwrite") \
    .jdbc(
        url=POSTGRES_URL,          
        table="lab.all_mock_data",
        properties=POSTGRES_PROPS,
    )

print("Таблица lab.all_mock_data заново создана в bigdata_lab.")

10000 50
+---+-------------------+------------------+------------+------------------------+----------------+--------------------+-----------------+-----------------+------------------+-----------------+----------------+--------------------------+------------------------+------------------+------------+----------------+-------------+----------------+----------+----------------+--------------+---------------+-------------+----------------+------------+--------------+---------------+-----------+-------------+------------+--------------------------+------------+--------------+-------------+------------+-------------+----------------+---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

In [4]:
table_query = "(SELECT * FROM lab.all_mock_data) AS all_mock_data"

df_raw = spark.read.jdbc(
    url=POSTGRES_URL,
    table=table_query,
    properties=POSTGRES_PROPS,
)

print(f"Строк: {df_raw.count()}, столбцов: {len(df_raw.columns)}")
df_raw.show(10, truncate=False)
df_raw.printSchema()

Строк: 10000, столбцов: 50
+---+-------------------+------------------+------------+------------------------+----------------+--------------------+-----------------+-----------------+------------------+-----------------+----------------+--------------------------+------------------------+------------------+------------+----------------+-------------+----------------+----------+----------------+--------------+---------------+-------------+----------------+------------+--------------+---------------+-----------+-------------+------------+--------------------------+------------+--------------+-------------+------------+-------------+----------------+---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

In [5]:
df = (
    df_raw
    .withColumn("sale_date", to_date(col("sale_date"), "yyyy-MM-dd"))
    .withColumn("product_price", col("product_price").cast("double"))
    .withColumn("sale_quantity", col("sale_quantity").cast("int"))
    .withColumn("sale_total_price", col("sale_total_price").cast("double"))
    .withColumn("product_rating", col("product_rating").cast("double"))
    .withColumn("product_reviews", col("product_reviews").cast("int"))
)

df.printSchema()
df.show(5, truncate=False)

root
 |-- id: integer (nullable = true)
 |-- customer_first_name: string (nullable = true)
 |-- customer_last_name: string (nullable = true)
 |-- customer_age: integer (nullable = true)
 |-- customer_email: string (nullable = true)
 |-- customer_country: string (nullable = true)
 |-- customer_postal_code: string (nullable = true)
 |-- customer_pet_type: string (nullable = true)
 |-- customer_pet_name: string (nullable = true)
 |-- customer_pet_breed: string (nullable = true)
 |-- seller_first_name: string (nullable = true)
 |-- seller_last_name: string (nullable = true)
 |-- seller_email: string (nullable = true)
 |-- seller_country: string (nullable = true)
 |-- seller_postal_code: string (nullable = true)
 |-- product_name: string (nullable = true)
 |-- product_category: string (nullable = true)
 |-- product_price: double (nullable = true)
 |-- product_quantity: integer (nullable = true)
 |-- sale_date: date (nullable = true)
 |-- sale_customer_id: integer (nullable = true)
 |-- sale

In [6]:
print("Пропуски по столбцам:")
for c in df.columns:
    print(c, df.filter(col(c).isNull()).count())

print("\nУникальные значения product_category:")
df.select("product_category").distinct().show()

print("\nДиапазон дат:")
df.select(F_min("sale_date"), F_max("sale_date")).show()

Пропуски по столбцам:
id 0
customer_first_name 0
customer_last_name 0
customer_age 0
customer_email 0
customer_country 0
customer_postal_code 5183
customer_pet_type 0
customer_pet_name 0
customer_pet_breed 0
seller_first_name 0
seller_last_name 0
seller_email 0
seller_country 0
seller_postal_code 5268
product_name 0
product_category 0
product_price 0
product_quantity 0
sale_date 10000
sale_customer_id 0
sale_seller_id 0
sale_product_id 0
sale_quantity 0
sale_total_price 0
store_name 0
store_location 0
store_city 0
store_state 8389
store_country 0
store_phone 0
store_email 0
pet_category 0
product_weight 0
product_color 0
product_size 0
product_brand 0
product_material 0
product_description 0
product_rating 0
product_reviews 0
product_release_date 0
product_expiry_date 0
supplier_name 0
supplier_contact 0
supplier_email 0
supplier_phone 0
supplier_address 0
supplier_city 0
supplier_country 0

Уникальные значения product_category:
+----------------+
|product_category|
+----------------+


In [7]:
from pyspark.sql.functions import (
    col,
    to_date,
    date_format,
    year,
    month,
    dayofmonth,
    weekofyear,
    row_number
)

from pyspark.sql.window import Window

In [8]:
dim_date = (
    df.select(to_date(col("sale_date")).alias("date"))
      .distinct()
)

dim_date = (
    dim_date
    .withColumn("date_id", date_format(col("date"), "yyyyMMdd").cast("int"))
    .withColumn("year", year("date"))
    .withColumn("month", month("date"))
    .withColumn("day", dayofmonth("date"))
    .withColumn("day_of_week", date_format("date", "E"))
    .withColumn("week_of_year", weekofyear("date"))
)

dim_date.show(5)

+----+-------+----+-----+----+-----------+------------+
|date|date_id|year|month| day|day_of_week|week_of_year|
+----+-------+----+-----+----+-----------+------------+
|null|   null|null| null|null|       null|        null|
+----+-------+----+-----+----+-----------+------------+



In [9]:
from pyspark.sql.functions import sha2


dim_product = (
    df.select(
        "product_name",
        "product_category",
        "product_brand",
        "pet_category"
    )
    .distinct()
    .withColumn("product_id", sha2(col("product_name"), 256))
)

dim_product.show(5)

+------------+----------------+-------------+------------+--------------------+
|product_name|product_category|product_brand|pet_category|          product_id|
+------------+----------------+-------------+------------+--------------------+
|   Bird Cage|            Cage|      Skalith|        Dogs|300d0c3fb7932d93e...|
|     Cat Toy|            Cage| Chatterpoint|        Fish|feee1f3e04b8207e9...|
|    Dog Food|            Cage|     Livetube|        Dogs|43188542d968bf9b3...|
|   Bird Cage|            Cage| Thoughtstorm|        Cats|300d0c3fb7932d93e...|
|   Bird Cage|             Toy|        Aivee|    Reptiles|300d0c3fb7932d93e...|
+------------+----------------+-------------+------------+--------------------+
only showing top 5 rows



In [10]:
dim_store = (
    df.select(
        "store_name",
        "store_city",
        "store_country"
    )
    .distinct()
    .withColumn("store_id", sha2(col("store_name"), 256))
)

dim_store.show(5)

+----------+----------+-------------+--------------------+
|store_name|store_city|store_country|            store_id|
+----------+----------+-------------+--------------------+
|     Yamia|    Cognac|         Peru|6e1e2d858212131d6...|
|Brainverse|   Bogoria|      Vietnam|9678b2e39105b3a9e...|
|     Quatz|    Spitak|       Russia|2b65bac86261a137f...|
|     Quinu|    Danzao|        Kenya|a2f65d668caf986d3...|
|  Jaxworks|Nowosielce|      Morocco|3e204e1f903d8e5f0...|
+----------+----------+-------------+--------------------+
only showing top 5 rows



In [11]:
dim_customer = (
    df.select(
        "customer_email",
        "customer_country"
    )
    .distinct()
    .withColumn("customer_id", sha2(col("customer_email"), 256))
)

dim_customer.show(5)

+--------------------+----------------+--------------------+
|      customer_email|customer_country|         customer_id|
+--------------------+----------------+--------------------+
|mberford2h@intel.com|   United States|26553b9f04b79b5f9...|
|pfarrar7o@busines...|           China|3616b1a79b85f17fc...|
|mhimpsoncn@comsen...|     North Korea|d8c4200ad41072a58...|
|smaryeth4@science...|          Greece|c4d3a492f989998fd...|
|vhawkswellit@berk...|       Indonesia|3fb1eb573640bb196...|
+--------------------+----------------+--------------------+
only showing top 5 rows



In [12]:
from pyspark.sql.functions import sha2

fact_sales = (
    df
    .withColumn("date_id", date_format(col("sale_date"), "yyyyMMdd").cast("int"))
    .withColumn("product_id", sha2(col("product_name"), 256))
    .withColumn("store_id", sha2(col("store_name"), 256))
    .withColumn("customer_id", sha2(col("customer_email"), 256))
    .select(
        "date_id",
        "product_id",
        "store_id",
        "customer_id",
        "sale_quantity",
        "sale_total_price"
    )
)

fact_sales.limit(5).show()   

+-------+--------------------+--------------------+--------------------+-------------+----------------+
|date_id|          product_id|            store_id|         customer_id|sale_quantity|sale_total_price|
+-------+--------------------+--------------------+--------------------+-------------+----------------+
|   null|300d0c3fb7932d93e...|52a2e2f0da77d91e0...|c28ac1e17277a6f14...|            3|           83.32|
|   null|43188542d968bf9b3...|af6bfa894e50ec1b4...|3755b19dec31bd3b2...|            4|           384.2|
|   null|43188542d968bf9b3...|231528c8dc5835247...|56a666227963c7ab7...|            6|           11.58|
|   null|feee1f3e04b8207e9...|797a7a81258d80494...|3fe1226e00e6559b5...|            2|          273.35|
|   null|300d0c3fb7932d93e...|fd582e081e04f7e3f...|d78231e3c603de9f5...|            2|          305.87|
+-------+--------------------+--------------------+--------------------+-------------+----------------+



In [13]:
(
    dim_date
    .select(
        "date_id",
        "date",
        "year",
        "month",
        "day",
        "day_of_week",
        "week_of_year"
    )
    .write
    .mode("overwrite")
    .jdbc(
        url=POSTGRES_URL,
        table="lab.dim_date",
        properties=POSTGRES_PROPS,
    )
)


(
    dim_product
    .select(
        "product_id",
        "product_name",
        "product_category",
        "product_brand",
        "pet_category"
    )
    .write
    .mode("overwrite")
    .jdbc(
        url=POSTGRES_URL,
        table="lab.dim_product",
        properties=POSTGRES_PROPS,
    )
)


(
    dim_store
    .select(
        "store_id",
        "store_name",
        "store_city",
        "store_country"
    )
    .write
    .mode("overwrite")
    .jdbc(
        url=POSTGRES_URL,
        table="lab.dim_store",
        properties=POSTGRES_PROPS,
    )
)


(
    dim_customer
    .select(
        "customer_id",
        "customer_email",
        "customer_country"
    )
    .write
    .mode("overwrite")
    .jdbc(
        url=POSTGRES_URL,
        table="lab.dim_customer",
        properties=POSTGRES_PROPS,
    )
)


(
    fact_sales
    .write
    .mode("overwrite")
    .jdbc(
        url=POSTGRES_URL,
        table="lab.fact_sales",
        properties=POSTGRES_PROPS,
    )
)

In [14]:
# === Читаем модель "звезда" из PostgreSQL и собираем единый факт ===

dim_date = spark.read.jdbc(POSTGRES_URL, "lab.dim_date", properties=POSTGRES_PROPS)
dim_product = spark.read.jdbc(POSTGRES_URL, "lab.dim_product", properties=POSTGRES_PROPS)
dim_store = spark.read.jdbc(POSTGRES_URL, "lab.dim_store", properties=POSTGRES_PROPS)
dim_customer = spark.read.jdbc(POSTGRES_URL, "lab.dim_customer", properties=POSTGRES_PROPS)
fact_sales = spark.read.jdbc(POSTGRES_URL, "lab.fact_sales", properties=POSTGRES_PROPS)

# ВАЖНО: проверь название поля с датой в dim_date.
# Ниже я использую date_value — замени на своё (например, "date" или "full_date").
fact_enriched = (
    fact_sales
    .join(dim_product, "product_id")
    .join(dim_store, "store_id")
    .join(dim_customer, "customer_id")
    .join(dim_date, "date_id")
)

fact_enriched.cache()
fact_enriched.show(5)

+-------+-----------+--------+----------+-------------+----------------+------------+----------------+-------------+------------+----------+----------+-------------+--------------+----------------+----+----+-----+---+-----------+------------+
|date_id|customer_id|store_id|product_id|sale_quantity|sale_total_price|product_name|product_category|product_brand|pet_category|store_name|store_city|store_country|customer_email|customer_country|date|year|month|day|day_of_week|week_of_year|
+-------+-----------+--------+----------+-------------+----------------+------------+----------------+-------------+------------+----------+----------+-------------+--------------+----------------+----+----+-----+---+-----------+------------+
+-------+-----------+--------+----------+-------------+----------------+------------+----------------+-------------+------------+----------+----------+-------------+--------------+----------------+----+----+-----+---+-----------+------------+



In [15]:
sales_by_product = (
    df.groupBy(
        "product_name",
        "product_category",
        "product_brand",
        "pet_category"
    )
    .agg(
        F_sum("sale_quantity").alias("total_quantity"),
        F_sum("sale_total_price").alias("total_revenue"),
        F_avg("product_price").alias("avg_price"),
        F_avg("product_rating").alias("avg_rating"),
        F_sum("product_reviews").alias("total_reviews"),
        countDistinct("customer_email").alias("unique_customers"),
        F_min("sale_date").alias("first_sale_date"),
        F_max("sale_date").alias("last_sale_date")
    )
)

sales_by_product.show(5)

+------------+----------------+-------------+------------+--------------+-------------+-----------------+------------------+-------------+----------------+---------------+--------------+
|product_name|product_category|product_brand|pet_category|total_quantity|total_revenue|        avg_price|        avg_rating|total_reviews|unique_customers|first_sale_date|last_sale_date|
+------------+----------------+-------------+------------+--------------+-------------+-----------------+------------------+-------------+----------------+---------------+--------------+
|     Cat Toy|             Toy|        Quimm|    Reptiles|             9|        36.56|            56.18|               1.5|          808|               1|           null|          null|
|    Dog Food|            Cage|     Riffpath|       Birds|             8|        94.02|            22.86|               1.8|          760|               1|           null|          null|
|    Dog Food|             Toy|        Rooxo|        Fish|       

In [16]:
from pyspark.sql.functions import col

top10_products_by_revenue = (
    sales_by_product
    .orderBy(col("total_revenue").desc())
    .limit(10)
)

top10_products_by_revenue.show(5)

+------------+----------------+-------------+------------+--------------+------------------+------------------+------------------+-------------+----------------+---------------+--------------+
|product_name|product_category|product_brand|pet_category|total_quantity|     total_revenue|         avg_price|        avg_rating|total_reviews|unique_customers|first_sale_date|last_sale_date|
+------------+----------------+-------------+------------+--------------+------------------+------------------+------------------+-------------+----------------+---------------+--------------+
|    Dog Food|             Toy|        Npath|       Birds|            37|1987.7199999999998| 77.63166666666667|3.0833333333333335|         4105|               6|           null|          null|
|   Bird Cage|            Cage|    Youbridge|        Dogs|            23|           1802.61|57.748000000000005|               3.5|         1944|               5|           null|          null|
|   Bird Cage|            Food|    

In [17]:
# Топ-10 продуктов по выручке
top10_products_by_revenue = sales_by_product.orderBy(col("total_revenue").desc()).limit(10)

# Выручка по категориям
revenue_by_category = (
    sales_by_product
    .groupBy("product_category")
    .agg(F_sum("total_revenue").alias("category_revenue"))
)

top10_products_by_revenue.show(10)
revenue_by_category.show(5)

+------------+----------------+-------------+------------+--------------+------------------+------------------+------------------+-------------+----------------+---------------+--------------+
|product_name|product_category|product_brand|pet_category|total_quantity|     total_revenue|         avg_price|        avg_rating|total_reviews|unique_customers|first_sale_date|last_sale_date|
+------------+----------------+-------------+------------+--------------+------------------+------------------+------------------+-------------+----------------+---------------+--------------+
|    Dog Food|             Toy|        Npath|       Birds|            37|1987.7199999999998| 77.63166666666667|3.0833333333333335|         4105|               6|           null|          null|
|   Bird Cage|            Cage|    Youbridge|        Dogs|            23|           1802.61|57.748000000000005|               3.5|         1944|               5|           null|          null|
|   Bird Cage|            Food|    

In [18]:
sales_by_store = (
    df.groupBy(
        "store_name",
        "store_city",
        "store_country"
    )
    .agg(
        F_sum("sale_quantity").alias("total_quantity"),
        F_sum("sale_total_price").alias("total_revenue"),
        F_count("*").alias("orders_count"),  # количество строк, считаем как число заказов
        countDistinct("product_name").alias("unique_products"),
        countDistinct("customer_email").alias("unique_customers"),
        F_min("sale_date").alias("first_sale_date"),
        F_max("sale_date").alias("last_sale_date")
    )
    .withColumn(
        "avg_check",
        col("total_revenue") / col("orders_count")
    )
)

sales_by_store.show(5)

+----------+-----------+--------------+--------------+-------------+------------+---------------+----------------+---------------+--------------+---------+
|store_name| store_city| store_country|total_quantity|total_revenue|orders_count|unique_products|unique_customers|first_sale_date|last_sale_date|avg_check|
+----------+-----------+--------------+--------------+-------------+------------+---------------+----------------+---------------+--------------+---------+
|     Quatz|     Spitak|        Russia|            10|       139.49|           1|              1|               1|           null|          null|   139.49|
|     Aimbo|San Antonio|      Malaysia|             5|       414.06|           1|              1|               1|           null|          null|   414.06|
|     Skiba|   Znamenka|     Indonesia|             1|       340.46|           1|              1|               1|           null|          null|   340.46|
|  Brainbox|      Banga|         China|             6|       290

In [19]:
top5_stores_by_revenue = (
    sales_by_store
    .orderBy(col("total_revenue").desc())
    .limit(5)
)

top5_stores_by_revenue.show(5)

+-----------+----------+-------------+--------------+-------------+------------+---------------+----------------+---------------+--------------+---------+
| store_name|store_city|store_country|total_quantity|total_revenue|orders_count|unique_products|unique_customers|first_sale_date|last_sale_date|avg_check|
+-----------+----------+-------------+--------------+-------------+------------+---------------+----------------+---------------+--------------+---------+
|       DabZ|    Grekan| South Africa|             7|       499.85|           1|              1|               1|           null|          null|   499.85|
|Thoughtblab|     Fonte|       Poland|             9|        499.8|           1|              1|               1|           null|          null|    499.8|
|     Camido| Longzhong|       Sweden|             2|       499.76|           1|              1|               1|           null|          null|   499.76|
|   Edgeblab|     Pesek|    Indonesia|             8|       499.76|   

In [20]:
sales_by_month = (
    df.withColumn("year", date_format(col("sale_date"), "yyyy"))
      .withColumn("month", date_format(col("sale_date"), "MM"))
      .withColumn("year_month", date_format(col("sale_date"), "yyyy-MM"))
      .groupBy("year", "month", "year_month")
      .agg(
          F_sum("sale_quantity").alias("total_quantity"),
          F_sum("sale_total_price").alias("total_revenue"),
          F_count("*").alias("orders_count"),
          countDistinct("customer_email").alias("unique_customers"),
          countDistinct("product_name").alias("unique_products")
      )
      .withColumn(
          "avg_order_value",
          col("total_revenue") / col("orders_count")
      )
)

sales_by_month.show(5)

+----+-----+----------+--------------+------------------+------------+----------------+---------------+------------------+
|year|month|year_month|total_quantity|     total_revenue|orders_count|unique_customers|unique_products|   avg_order_value|
+----+-----+----------+--------------+------------------+------------+----------------+---------------+------------------+
|null| null|      null|         54623|2529852.1199999987|       10000|           10000|              3|252.98521199999988|
+----+-----+----------+--------------+------------------+------------+----------------+---------------+------------------+



In [21]:
sales_by_customer = (
    df.groupBy(
        "customer_email",
        "customer_country"
    )
    .agg(
        F_sum("sale_total_price").alias("total_revenue"),
        F_sum("sale_quantity").alias("items_bought"),
        F_count("*").alias("orders_count"),
        countDistinct("product_name").alias("unique_products"),
        F_min("sale_date").alias("first_purchase"),
        F_max("sale_date").alias("last_purchase"),
        F_avg("product_rating").alias("avg_rating_preference")
    )
    .withColumn(
        "avg_check",
        col("total_revenue") / col("orders_count")
    )
)

sales_by_customer.show(5)

+--------------------+----------------+-------------+------------+------------+---------------+--------------+-------------+---------------------+---------+
|      customer_email|customer_country|total_revenue|items_bought|orders_count|unique_products|first_purchase|last_purchase|avg_rating_preference|avg_check|
+--------------------+----------------+-------------+------------+------------+---------------+--------------+-------------+---------------------+---------+
|kspatonioo@google.es|       Indonesia|       355.51|           7|           1|              1|          null|         null|                  2.7|   355.51|
|     pargueb1@i2i.jp|          Brazil|       154.64|           1|           1|              1|          null|         null|                  2.6|   154.64|
|smaryeth4@science...|          Greece|       393.81|           8|           1|              1|          null|         null|                  2.6|   393.81|
|    rmeertjx@com.com|         Somalia|        180.8|     

In [22]:
top10_customers_by_revenue = (
    sales_by_customer
    .orderBy(col("total_revenue").desc())
    .limit(10)
)

top10_customers_by_revenue.show(10)

+--------------------+----------------+-------------+------------+------------+---------------+--------------+-------------+---------------------+---------+
|      customer_email|customer_country|total_revenue|items_bought|orders_count|unique_products|first_purchase|last_purchase|avg_rating_preference|avg_check|
+--------------------+----------------+-------------+------------+------------+---------------+--------------+-------------+---------------------+---------+
| bfeasby57@youku.com|         Albania|       499.85|           7|           1|              1|          null|         null|                  4.5|   499.85|
|sstappardbp@busin...|        Portugal|        499.8|           9|           1|              1|          null|         null|                  4.5|    499.8|
|dsorea0@geocities...|           China|       499.76|           2|           1|              1|          null|         null|                  4.8|   499.76|
|    rivattspm@un.org|       Indonesia|       499.76|     

In [23]:
supplier_stats_by_brand = (
    df.groupBy("product_brand")
    .agg(
        F_sum("sale_total_price").alias("brand_revenue"),
        F_sum("sale_quantity").alias("brand_quantity"),
        countDistinct("product_name").alias("unique_products"),
        countDistinct("store_name").alias("unique_stores"),
        F_avg("product_price").alias("avg_product_price"),
        F_min("sale_date").alias("first_sale_date"),
        F_max("sale_date").alias("last_sale_date")
    )
)

supplier_stats_by_brand.show(5)

+-------------+------------------+--------------+---------------+-------------+------------------+---------------+--------------+
|product_brand|     brand_revenue|brand_quantity|unique_products|unique_stores| avg_product_price|first_sale_date|last_sale_date|
+-------------+------------------+--------------+---------------+-------------+------------------+---------------+--------------+
|      Jetwire| 8208.589999999998|           134|              3|           24| 38.58962962962963|           null|          null|
|     Jaxworks| 6771.570000000001|           140|              3|           24| 43.73079999999999|           null|          null|
|    Reallinks| 8587.970000000003|           186|              3|           31| 49.18030303030304|           null|          null|
|  Brainlounge| 7367.210000000002|           166|              3|           25|47.835384615384605|           null|          null|
|     Snaptags|3501.5799999999995|            97|              3|           17| 53.4052941

In [24]:
supplier_stats_by_brand_country = (
    df.groupBy("product_brand", "store_country")
    .agg(
        F_sum("sale_total_price").alias("brand_revenue"),
        F_sum("sale_quantity").alias("brand_quantity")
    )
)

supplier_stats_by_brand_country.show(5)

+-------------+-------------+-----------------+--------------+
|product_brand|store_country|    brand_revenue|brand_quantity|
+-------------+-------------+-----------------+--------------+
|       Skivee|       Russia|           347.85|             8|
|       Talane|      Finland|            199.8|             4|
|        Quaxo|       Brazil|          1040.88|            31|
|   Zoomlounge|        China|832.3999999999999|            35|
|     Edgeclub|        China|          1435.16|            35|
+-------------+-------------+-----------------+--------------+
only showing top 5 rows



In [25]:
from pyspark.sql.functions import sum as F_sum

top5_suppliers_by_revenue = (
    supplier_stats_by_brand
    .orderBy(col("brand_revenue").desc())   # или как у тебя называется выручка по бренду
    .limit(5)
)

top5_suppliers_by_revenue.show(5)

+-------------+------------------+--------------+---------------+-------------+------------------+---------------+--------------+
|product_brand|     brand_revenue|brand_quantity|unique_products|unique_stores| avg_product_price|first_sale_date|last_sale_date|
+-------------+------------------+--------------+---------------+-------------+------------------+---------------+--------------+
|         Jayo|17530.880000000005|           358|              3|           57|50.844374999999985|           null|          null|
|      Dynabox|15270.049999999994|           301|              3|           49| 54.92094339622644|           null|          null|
|     Photobug|          14842.76|           305|              3|           52|49.590727272727285|           null|          null|
| Thoughtstorm|14269.489999999998|           275|              3|           49|51.573018867924524|           null|          null|
|      Youspan|14268.220000000003|           311|              3|           54|46.18964285

In [26]:
quality_by_product = (
    df.groupBy(
        "product_name",
        "product_category",
        "product_brand",
        "pet_category"
    )
    .agg(
        F_avg("product_rating").alias("avg_rating"),
        F_sum("product_reviews").alias("total_reviews"),
        F_sum("sale_quantity").alias("total_quantity"),
        F_sum("sale_total_price").alias("total_revenue")
    )
)

quality_by_product.show(5)

+------------+----------------+-------------+------------+------------------+-------------+--------------+-----------------+
|product_name|product_category|product_brand|pet_category|        avg_rating|total_reviews|total_quantity|    total_revenue|
+------------+----------------+-------------+------------+------------------+-------------+--------------+-----------------+
|   Bird Cage|            Cage|      Skalith|        Dogs|               3.7|         1761|             5|           213.91|
|     Cat Toy|            Cage| Chatterpoint|        Fish|3.9000000000000004|         1139|             7|           760.47|
|    Dog Food|            Cage|     Livetube|        Dogs|               4.3|          932|             5|           192.43|
|   Bird Cage|            Cage| Thoughtstorm|        Cats|              3.75|         1779|            17|           763.02|
|   Bird Cage|             Toy|        Aivee|    Reptiles|2.1500000000000004|          964|            12|819.8499999999999|


In [27]:
from pyspark.sql.functions import corr

corr_rating_quantity = (
    quality_by_product
    .select(corr("avg_rating", "total_quantity").alias("corr_rating_quantity"))
    .collect()[0]["corr_rating_quantity"]
)

print("Корреляция между средним рейтингом и объёмом продаж:", corr_rating_quantity)

Корреляция между средним рейтингом и объёмом продаж: 0.006655472092664356


In [28]:
from pymongo import MongoClient

mongo = MongoClient(MONGO_URI)
db = mongo.bigdata_lab

# Удаляем старые коллекции
db.sales_by_product.drop()
db.sales_by_store.drop()
db.sales_by_customer.drop()
db.sales_by_month.drop()
db.supplier_stats_by_brand.drop()
db.quality_by_product.drop()
db.top10_products_by_revenue.drop()
db.top5_stores_by_revenue.drop()
db.top10_customers_by_revenue.drop()
db.top5_suppliers_by_revenue.drop()

# Загружаем финальные витрины
db.sales_by_product.insert_many(sales_by_product.toPandas().to_dict("records"))
db.sales_by_store.insert_many(sales_by_store.toPandas().to_dict("records"))
db.sales_by_customer.insert_many(sales_by_customer.toPandas().to_dict("records"))
db.sales_by_month.insert_many(sales_by_month.toPandas().to_dict("records"))
db.supplier_stats_by_brand.insert_many(supplier_stats_by_brand.toPandas().to_dict("records"))
db.quality_by_product.insert_many(quality_by_product.toPandas().to_dict("records"))
db.top10_products_by_revenue.insert_many(top10_products_by_revenue.toPandas().to_dict("records"))
db.top5_stores_by_revenue.insert_many(top5_stores_by_revenue.toPandas().to_dict("records"))
db.top10_customers_by_revenue.insert_many(top10_customers_by_revenue.toPandas().to_dict("records"))
db.top5_suppliers_by_revenue.insert_many(top5_suppliers_by_revenue.toPandas().to_dict("records"))

print("MongoDB: финальные витрины успешно записаны!")

MongoDB: финальные витрины успешно записаны!


In [29]:
from clickhouse_driver import Client

CH_HOST = "clickhouse"
CH_PORT = 9000
CH_USER = "labuser"
CH_PASS = "labpass"
CH_DB   = "analytics"

client_ch = Client(
    host=CH_HOST,
    port=CH_PORT,
    user=CH_USER,
    password=CH_PASS
)

client_ch.execute(f"CREATE DATABASE IF NOT EXISTS {CH_DB}")

def ch_insert(df, table_name):
    pdf = df.toPandas()
    client_ch.execute(f"DROP TABLE IF EXISTS {CH_DB}.{table_name}")
    columns_def = ", ".join([f"`{col}` String" for col in pdf.columns])
    create_sql = f"""
        CREATE TABLE {CH_DB}.{table_name} (
            {columns_def}
        )
        ENGINE = MergeTree()
        ORDER BY tuple()
    """
    client_ch.execute(create_sql)
    data = [
        tuple('' if x is None else str(x) for x in row)
        for row in pdf.to_numpy()
    ]
    insert_sql = f"""
        INSERT INTO {CH_DB}.{table_name} ({", ".join(f"`{c}`" for c in pdf.columns)})
        VALUES
    """
    client_ch.execute(insert_sql, data)

ModuleNotFoundError: No module named 'clickhouse_driver'

In [ ]:
ch_insert(sales_by_product,"sales_by_product")
ch_insert(sales_by_store,"sales_by_store")
ch_insert(sales_by_customer,"sales_by_customer")
ch_insert(sales_by_month,"sales_by_month")
ch_insert(supplier_stats_by_brand,"supplier_stats_by_brand")
ch_insert(quality_by_product,"quality_by_product")
ch_insert(top10_products_by_revenue, "top10_products_by_revenue")
ch_insert(top5_stores_by_revenue, "top5_stores_by_revenue")
ch_insert(top10_customers_by_revenue, "top10_customers_by_revenue")
ch_insert(top5_suppliers_by_revenue, "top5_suppliers_by_revenue")

print("✔ Все финальные витрины успешно загружены в ClickHouse!")